In [1]:
from langchain_aws import ChatBedrockConverse

In [2]:
llm = ChatBedrockConverse(model_id = 'cohere.command-r-plus-v1:0', region_name = 'us-east-1', temperature = 0.5, max_tokens = 300)

In [4]:
llm.invoke('Explain deep learning in 3-4 lines').content

"Deep learning is a subset of machine learning that trains a neural network with multiple layers of interconnected nodes, mimicking the human brain's structure and functionality. It uses large amounts of data and multiple processing layers to learn and make accurate predictions or decisions. This technique has revolutionized various fields, including image and speech recognition, natural language processing, and autonomous systems."

### Messages

In [5]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [6]:
messages = [SystemMessage("You are a financial expert. Recommend investment strategies to the user"), 
            HumanMessage("Tell me best investment strategies")] 
messages

[SystemMessage(content='You are a financial expert. Recommend investment strategies to the user', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Tell me best investment strategies', additional_kwargs={}, response_metadata={})]

In [7]:
from langchain_core.output_parsers import StrOutputParser
chain = llm | StrOutputParser()
chain.invoke(messages)

'There are many investment strategies that can be successful, depending on your financial goals and risk tolerance. Here are a few common strategies that are often recommended:\n\n1. Buy-and-hold: This strategy involves buying stocks, bonds, or other investments and holding them for the long term, regardless of market fluctuations. This strategy is based on the assumption that the value of the investments will increase over time, and it can be a good way to build wealth over the long term.\n\n2. Dollar-cost averaging: This strategy involves investing a fixed amount of money on a regular basis, regardless of the market price of the investment. By investing a fixed amount, you buy more shares when prices are low and fewer shares when prices are high. This can help to reduce the impact of market volatility on your investment portfolio.\n\n3. Diversification: This strategy involves spreading your investments across a variety of asset classes, sectors, and geographic regions. By diversifyin

### Document

In [8]:
from langchain_core.documents import Document

In [9]:
documents = Document(page_content = "This is some random text from the user to show an example on langchain documents", 
         metadata = {"id":43, "source":"Some random source", "source_id":547}) 
documents

Document(metadata={'id': 43, 'source': 'Some random source', 'source_id': 547}, page_content='This is some random text from the user to show an example on langchain documents')

In [10]:
type(documents)

langchain_core.documents.base.Document

### Prompt Templates
#### string prompt template

In [11]:
from langchain_core.prompts import PromptTemplate 

In [12]:
#Example 1
chat = PromptTemplate.from_template("Let's have a debate on the topic: {topic}") 
res = chat.invoke({"topic":"Geo-politics"})
llm.invoke(res).content

"Sure! I'd be happy to participate in a debate on the topic of geo-politics. \n\nFor the purpose of this debate, let's define geo-politics as the study of the impact of geographical factors on international politics and the interactions between nations. This includes considerations of territory, resources, demographics, and the international balance of power. \n\nHere is a potential structure for our debate, with me taking the position arguing for the importance and relevance of geo-politics: \n\n## Opening Statements: \n\n**Argument For:** \nGeo-politics is an essential lens through which to view and understand international relations. It provides a framework for analyzing how geographical factors influence the behavior of nations and the dynamics between them. By considering the distribution of resources, the configuration of territories, and the interplay of diverse cultures and populations, geo-politics offers insights into the motivations and strategies of nations on the world sta

In [13]:
#Example 2
prompt = PromptTemplate.from_template("Tell me a fun fact about the city in 2-3 lines: {city}") 
res = prompt.invoke({"city":"Jaipur"}) 
llm.invoke(res).content

'Jaipur, also known as the Pink City, was painted pink in 1876 to welcome the Prince of Wales and the color has since become a symbol of the city. The vibrant color was chosen as it was traditionally associated with hospitality.'

#### Chat Prompt Template

In [14]:
from langchain_core.prompts import ChatPromptTemplate

In [15]:
#Example -1
chat = ChatPromptTemplate.from_messages([("system","You are friendly AI assistant. Answer in 2-3 lines on what user asks"), 
                           ("user","Define the following finance term:{finance_term}")]) 
res = chat.invoke({'finance_term':'repo_rate'})
llm.invoke(res).content

'Repo rate, or repurchase rate, is the interest rate at which a central bank lends money to commercial banks in case of a shortfall of funds. It is a tool used by central banks to control liquidity in the banking system and the economy.'

In [16]:
#Example -2
template = ChatPromptTemplate.from_messages([ 
    ("system","You are a helpful assistant that translates {input_language} to {output_language}"), 
    ("human", "Translate this sentence: {text}")]) 

llm.invoke(template.invoke({'input_language':'English', 'output_language':'Hinglish', 'text':'Good Morning, how are you ?'})).content

'"Gud morming, hao aar yuu?"'

In [17]:
# Example -3 
prompt_temp = ChatPromptTemplate.from_messages([ 
    ("system", "You are a grumpy historian who hates tourists and loves {some_input}"), 
    ("human","Tell me something funny related to {city}")])
res = prompt_temp.invoke({"some_input":"Samosa","city":"Ladak"})
llm.invoke(res).content

'As a grumpy historian who has seen too many tourists, I\'ll share a tale that might make you chuckle. It\'s about a group of tourists who wanted a "authentic local experience" in Ladakh.\n\nSo these tourists, decked out in their brand new hiking gear and selfie sticks, decided they wanted to immerse themselves in the local culture. They heard about a remote village known for its traditional way of life, untouched by modern influences. Eager for an adventure, they set off on a journey to find this village.\n\nAs they trekked through the breathtaking landscape, they felt a sense of accomplishment, believing they were embracing the true spirit of Ladakh. When they finally reached the village, they were greeted by a group of locals herding yaks. The tourists were excited, thinking they had stumbled upon a living museum.\n\nOne of the tourists, let\'s call him Bob, approached an elderly herder and asked, "Oh wise sir, can you share with us the secrets of your ancient ways? We seek a genuin

### Messages Placeholder

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder 
from langchain_core.messages import HumanMessage, AIMessage

In [4]:
#Example -1
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant named Jarvis"),
    MessagesPlaceholder(variable_name = "chat_history"), 
    ("human","{input}")])

In [33]:
history = [
    HumanMessage(content='Hi, my name is Tony Stark'), 
    AIMessage(content = "Hello Mr.Stark. How can I assist you today")
]

In [21]:
res = prompt.invoke({'chat_history':history, 'input':'Wait, what did I say my name was?'})
llm.invoke(res).content

'You said your name was Tony Stark.'

In [5]:
#Example -2 Verify whether the history is being stored
history = []
def chat_with_verification(info): 
    global history 
    res = llm.invoke(prompt.invoke({'chat_history':history, 'input':info}))
    history.append(HumanMessage(content = info)) 
    history.append(AIMessage(content = res.content))
    print(f"\n> User: {info}") 
    print(f"> AI: {res.content}")
    for i, msg in enumerate(history): 
        print(f" Message {i}: {msg.type} -> {msg.content[:20]}...")

def clear_memory(): 
    global history 
    history = [] 
    print("------Memory Cleared-------")

In [6]:
chat_with_verification("Hi, I'm Nick")


> User: Hi, I'm Nick
> AI: Hi Nick, I'm Jarvis. How can I help you today?
 Message 0: human -> Hi, I'm Nick...
 Message 1: ai -> Hi Nick, I'm Jarvis....


In [7]:
chat_with_verification("Hi I'm Tony")


> User: Hi I'm Tony
> AI: Hi Tony, I'm Jarvis. It's nice to meet you. What can I do for you?
 Message 0: human -> Hi, I'm Nick...
 Message 1: ai -> Hi Nick, I'm Jarvis....
 Message 2: human -> Hi I'm Tony...
 Message 3: ai -> Hi Tony, I'm Jarvis....


In [8]:
chat_with_verification("What was my again?")


> User: What was my again?
> AI: Your name is Tony.
 Message 0: human -> Hi, I'm Nick...
 Message 1: ai -> Hi Nick, I'm Jarvis....
 Message 2: human -> Hi I'm Tony...
 Message 3: ai -> Hi Tony, I'm Jarvis....
 Message 4: human -> What was my again?...
 Message 5: ai -> Your name is Tony....


In [9]:
clear_memory()

------Memory Cleared-------


In [10]:
chat_with_verification("What is my name?")


> User: What is my name?
> AI: Your name is Jarvis.
 Message 0: human -> What is my name?...
 Message 1: ai -> Your name is Jarvis....
